**Running on Colab with a T4 GPU:** choose *Runtime → Change runtime type → T4 GPU*,
then *Runtime → Run all*. The GPU check after the imports confirms the T4 is in use.
The whole notebook (the main training run plus the three exercise solutions) takes
roughly 15–25 minutes on a T4.

The following additional libraries are needed to run this
notebook. Note that running on Colab is experimental, please report a Github
issue if you have any problem.

**Fix for the install error:** `pip install d2l==1.0.3` fails on current Colab
(Python 3.12) and other recent Python versions. d2l 1.0.3 pins old releases of
its dependencies (`numpy==1.23.5`, `scipy==1.10.1`, `pandas==2.0.3`, ...) that
have no builds for these Python versions, so pip tries to compile them from source
and stops with an error. Installing d2l with `--no-deps` and using current versions
of its dependencies avoids this. pip may still print a warning that the pinned
versions don't match; that warning is harmless.

In [ ]:
# d2l 1.0.3 pins old dependency versions that can't be installed on Python 3.12+,
# so install d2l on its own and use current versions of its dependencies.
%pip install -q d2l==1.0.3 --no-deps
%pip install -q torch torchvision matplotlib matplotlib-inline pandas requests scipy

# Transformers for Vision

The Transformer architecture was initially proposed
for sequence-to-sequence learning,
with a focus on machine translation.
Subsequently, Transformers emerged as the model of choice
in various natural language processing tasks (Radford et al., 2018; Radford et al., 2019; Brown et al., 2020; Devlin et al., 2018; Raffel et al., 2020).
However, in the field of computer vision
the dominant architecture has remained
the CNN (see the chapter on modern CNNs).
Naturally, researchers started to wonder
if it might be possible to do better
by adapting Transformer models to image data.
This question sparked immense interest
in the computer vision community.
Recently, Ramachandran et al. (2019) proposed
a scheme for replacing convolution with self-attention.
However, its use of specialized patterns in attention
makes it hard to scale up models on hardware accelerators.
Then, Cordonnier et al. (2020) theoretically proved
that self-attention can learn to behave similarly to convolution.
Empirically, $2 \times 2$ patches were taken from images as inputs,
but the small patch size makes the model
only applicable to image data with low resolutions.

Without specific constraints on patch size,
*vision Transformers* (ViTs)
extract patches from images
and feed them into a Transformer encoder
to obtain a global representation,
which will finally be transformed for classification (Dosovitskiy et al., 2021).
Notably, Transformers show better scalability than CNNs:
when training larger models on larger datasets,
vision Transformers outperform ResNets by a significant margin.
Similar to the landscape of network architecture design in natural language processing,
Transformers have also become a game-changer in computer vision.


In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
# Check that the GPU is visible (on Colab this should print 'Tesla T4')
if torch.cuda.is_available():
    print('Using GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU found. On Colab choose Runtime > Change runtime type > T4 GPU, '
          'then run all cells again. (Training on CPU will be very slow.)')

## Model

The figure below depicts
the model architecture of vision Transformers.
This architecture consists of a stem
that patchifies images,
a body based on the multilayer Transformer encoder,
and a head that transforms the global representation
into the output label.

![The vision Transformer architecture. In this example, an image is split into nine patches. A special “&lt;cls&gt;” token and the nine flattened image patches are transformed via patch embedding and $\mathit{n}$ Transformer encoder blocks into ten representations, respectively. The “&lt;cls&gt;” representation is further transformed into the output label.](https://d2l.ai/_images/vit.svg)

Consider an input image with height $h$, width $w$,
and $c$ channels.
Specifying the patch height and width both as $p$,
the image is split into a sequence of $m = hw/p^2$ patches,
where each patch is flattened to a vector of length $cp^2$.
In this way, image patches can be treated similarly to tokens in text sequences by Transformer encoders.
A special “&lt;cls&gt;” (class) token and
the $m$ flattened image patches are linearly projected
into a sequence of $m+1$ vectors,
summed with learnable positional embeddings.
The multilayer Transformer encoder
transforms $m+1$ input vectors
into the same number of output vector representations of the same length.
It works exactly the same way as the original Transformer encoder,
only differing in the position of normalization.
Since the “&lt;cls&gt;” token attends to all the image patches
via self-attention,
its representation from the Transformer encoder output
will be further transformed into the output label.

## Patch Embedding

To implement a vision Transformer, let's start
with patch embedding in the figure above.
Splitting an image into patches
and linearly projecting these flattened patches
can be simplified as a single convolution operation,
where both the kernel size and the stride size are set to the patch size.


In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=96, patch_size=16, num_hiddens=512):
        super().__init__()
        def _make_tuple(x):
            if not isinstance(x, (list, tuple)):
                return (x, x)
            return x
        img_size, patch_size = _make_tuple(img_size), _make_tuple(patch_size)
        self.num_patches = (img_size[0] // patch_size[0]) * (
            img_size[1] // patch_size[1])
        self.conv = nn.LazyConv2d(num_hiddens, kernel_size=patch_size,
                                  stride=patch_size)

    def forward(self, X):
        # Output shape: (batch size, no. of patches, no. of channels)
        return self.conv(X).flatten(2).transpose(1, 2)

In the following example, taking images with height and width of `img_size` as inputs,
the patch embedding outputs `(img_size//patch_size)**2` patches
that are linearly projected to vectors of length `num_hiddens`.


In [ ]:
img_size, patch_size, num_hiddens, batch_size = 96, 16, 512, 4
patch_emb = PatchEmbedding(img_size, patch_size, num_hiddens)
X = torch.zeros(batch_size, 3, img_size, img_size)
d2l.check_shape(patch_emb(X),
                (batch_size, (img_size//patch_size)**2, num_hiddens))

## Vision Transformer Encoder

The MLP of the vision Transformer encoder is slightly different
from the positionwise FFN of the original Transformer encoder.
First, here the activation function uses the Gaussian error linear unit (GELU),
which can be considered as a smoother version of the ReLU (Hendrycks and Gimpel, 2016).
Second, dropout is applied to the output of each fully connected layer in the MLP for regularization.


In [ ]:
class ViTMLP(nn.Module):
    def __init__(self, mlp_num_hiddens, mlp_num_outputs, dropout=0.5):
        super().__init__()
        self.dense1 = nn.LazyLinear(mlp_num_hiddens)
        self.gelu = nn.GELU()
        self.dropout1 = nn.Dropout(dropout)
        self.dense2 = nn.LazyLinear(mlp_num_outputs)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout2(self.dense2(self.dropout1(self.gelu(
            self.dense1(x)))))

The vision Transformer encoder block implementation
just follows the pre-normalization design in the figure above,
where normalization is applied right *before* multi-head attention or the MLP.
In contrast to post-normalization ("add & norm" in the original Transformer),
where normalization is placed right *after* residual connections,
pre-normalization leads to more effective or efficient training for Transformers (Baevski and Auli, 2018; Wang et al., 2019; Xiong et al., 2020).


In [ ]:
class ViTBlock(nn.Module):
    def __init__(self, num_hiddens, norm_shape, mlp_num_hiddens,
                 num_heads, dropout, use_bias=False):
        super().__init__()
        self.ln1 = nn.LayerNorm(norm_shape)
        self.attention = d2l.MultiHeadAttention(num_hiddens, num_heads,
                                                dropout, use_bias)
        self.ln2 = nn.LayerNorm(norm_shape)
        self.mlp = ViTMLP(mlp_num_hiddens, num_hiddens, dropout)

    def forward(self, X, valid_lens=None):
        X = X + self.attention(*([self.ln1(X)] * 3), valid_lens)
        return X + self.mlp(self.ln2(X))

Just as in the original Transformer encoder,
no vision Transformer encoder block changes its input shape.


In [ ]:
X = torch.ones((2, 100, 24))
encoder_blk = ViTBlock(24, 24, 48, 8, 0.5)
encoder_blk.eval()
d2l.check_shape(encoder_blk(X), X.shape)

## Putting It All Together

The forward pass of vision Transformers below is straightforward.
First, input images are fed into a `PatchEmbedding` instance,
whose output is concatenated with the “&lt;cls&gt;”  token embedding.
They are summed with learnable positional embeddings before dropout.
Then the output is fed into the Transformer encoder that stacks `num_blks` instances of the `ViTBlock` class.
Finally, the representation of the “&lt;cls&gt;”  token is projected by the network head.


In [ ]:
class ViT(d2l.Classifier):
    """Vision Transformer."""
    def __init__(self, img_size, patch_size, num_hiddens, mlp_num_hiddens,
                 num_heads, num_blks, emb_dropout, blk_dropout, lr=0.1,
                 use_bias=False, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        self.patch_embedding = PatchEmbedding(
            img_size, patch_size, num_hiddens)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hiddens))
        num_steps = self.patch_embedding.num_patches + 1  # Add the cls token
        # Positional embeddings are learnable
        self.pos_embedding = nn.Parameter(
            torch.randn(1, num_steps, num_hiddens))
        self.dropout = nn.Dropout(emb_dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"{i}", ViTBlock(
                num_hiddens, num_hiddens, mlp_num_hiddens,
                num_heads, blk_dropout, use_bias))
        self.head = nn.Sequential(nn.LayerNorm(num_hiddens),
                                  nn.Linear(num_hiddens, num_classes))

    def forward(self, X):
        X = self.patch_embedding(X)
        X = torch.cat((self.cls_token.expand(X.shape[0], -1, -1), X), 1)
        X = self.dropout(X + self.pos_embedding)
        for blk in self.blks:
            X = blk(X)
        return self.head(X[:, 0])

## Training

Training a vision Transformer on the Fashion-MNIST dataset is just like how CNNs were trained in the chapter on modern CNNs.


In [ ]:
img_size, patch_size = 96, 16
num_hiddens, mlp_num_hiddens, num_heads, num_blks = 512, 2048, 8, 2
emb_dropout, blk_dropout, lr = 0.1, 0.1, 0.1
model = ViT(img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads,
            num_blks, emb_dropout, blk_dropout, lr)
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
data = d2l.FashionMNIST(batch_size=128, resize=(img_size, img_size))
trainer.fit(model, data)

## Summary and Discussion

You may have noticed that for small datasets like Fashion-MNIST,
our implemented vision Transformer
does not outperform the ResNet trained earlier in the book.
Similar observations can be made even on the ImageNet dataset (1.2 million images).
This is because Transformers *lack* those useful principles in convolution,
such as translation equivariance and locality.
However, the picture changes when training larger models on larger datasets (e.g., 300 million images),
where vision Transformers outperform ResNets by a large margin in image classification, showing
that Transformers scale better with more data and larger models (Dosovitskiy et al., 2021).
The introduction of vision Transformers
has changed the landscape of network design for modeling image data.
They were soon shown to be effective on the ImageNet dataset
with data-efficient training strategies of DeiT (Touvron et al., 2021).
However, the quadratic complexity of self-attention
in the number of patches
makes the Transformer architecture
less suitable for higher-resolution images.
Towards a general-purpose backbone network in computer vision,
Swin Transformers addressed the quadratic computational complexity
with respect to image size by computing attention within local windows
and reinstated convolution-like priors,
extending the applicability of Transformers to a range of computer vision tasks
beyond image classification with state-of-the-art results (Liu et al., 2021).

## Exercises

1. How does the value of `img_size` affect training time?
1. Instead of projecting the “&lt;cls&gt;” token representation to the output, how would you project the averaged patch representations? Implement this change and see how it affects the accuracy.
1. Can you modify hyperparameters to improve the accuracy of the vision Transformer?

The solutions follow below.


## Exercise Solutions

The cells below answer the three exercises. Run the notebook from the top first:
the solutions reuse `ViT`, the trained `model`, and `data` from the training cell.

### Exercise 1: How does `img_size` affect training time?

The image is cut into $m = (s/p)^2$ patches, where $s$ is `img_size` and $p$ is
`patch_size`, and the encoder processes $m + 1$ tokens (the patches plus `<cls>`).
With hidden size $d$, each encoder block costs roughly:

- $O(m\,d^2)$ for the linear layers: the query, key, value and output projections and the MLP;
- $O(m^2 d)$ for self-attention itself: the $m \times m$ attention scores and the weighted sum.

So the cost depends on the number of patches, which grows with the **square** of `img_size`:

- **Moderate image sizes (our case):** the linear layers dominate, so training time grows
  roughly in proportion to $m$, that is, with the **square of `img_size`**. Doubling
  `img_size` from 96 to 192 turns 36 patches into 144 and makes each step about 4× slower.
  With $d = 512$ and an MLP width of 2048, the attention term only becomes the larger one
  beyond roughly 3,000 patches.
- **Very large images:** once attention dominates, time grows with $m^2$, the **fourth power
  of `img_size`**. This is the quadratic cost of self-attention mentioned in the summary,
  and the reason plain ViTs are expensive on high-resolution images.
- **Memory** follows the same pattern, so larger images also force smaller batches.
- **On a GPU, very small images** may barely change the measured time, because the GPU is
  underused and fixed costs (data loading, kernel launches) dominate.

Two practical consequences for this notebook:

- Fashion-MNIST images are only 28×28 pixels. Resizing them to 96×96 adds no information,
  only cost: about 12× as many pixels to process.
- `img_size` should be divisible by `patch_size`. Otherwise the strided convolution in
  `PatchEmbedding` silently drops the leftover border pixels (for example, `img_size=100`
  with `patch_size=16` ignores the last 4 rows and columns).

The cell below measures the time per training step for several values of `img_size`,
keeping everything else fixed, and estimates the exponent $k$ in time $\propto$ `img_size`$^k$.

In [ ]:
import time
import numpy as np

def time_per_step(img_size, patch_size=16, batch_size=128, num_steps=10):
    """Average time of one training step (forward, backward, update) on random data."""
    device = d2l.try_gpu()
    net = ViT(img_size, patch_size, num_hiddens=512, mlp_num_hiddens=2048,
              num_heads=8, num_blks=2, emb_dropout=0.1, blk_dropout=0.1).to(device)
    X = torch.randn(batch_size, 1, img_size, img_size, device=device)
    y = torch.randint(0, 10, (batch_size,), device=device)
    net(X)  # initialize the lazy layers before creating the optimizer
    optimizer = torch.optim.SGD(net.parameters(), lr=0.1)

    def step():
        optimizer.zero_grad()
        net.loss(net(X), y).backward()
        optimizer.step()

    for _ in range(3):  # warm-up steps are not timed
        step()
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(num_steps):
        step()
    if device.type == 'cuda':
        torch.cuda.synchronize()  # wait for the GPU to finish before stopping the clock
    return (time.perf_counter() - start) / num_steps

sizes = [32, 64, 96, 128, 160, 192]
times = []
for s in sizes:
    times.append(time_per_step(s))
    print(f'img_size={s:3d}  patches={(s // 16) ** 2:3d}  '
          f'time per step={times[-1] * 1000:7.1f} ms')

# Slope of log(time) against log(img_size) over the larger sizes
k = np.polyfit(np.log(sizes[2:]), np.log(times[2:]), 1)[0]
print(f'time grows roughly like img_size ** {k:.1f}')
d2l.plot(sizes, [times], xlabel='img_size', ylabel='seconds per step',
         figsize=(4.5, 3))

### Exercise 2: Classify from the averaged patch representations

Instead of passing the encoder output at the `<cls>` position to the head, we can average
the encoder outputs of all patches (global average pooling) and pass that average to the
head. The `<cls>` token is then no longer needed, so the subclass below removes it and keeps
only the positional embeddings of the patches. The patch embedding, encoder blocks and head
are inherited from `ViT` unchanged; only the last line of `forward` really differs:

```python
return self.head(X[:, 0])        # <cls> token (original)
return self.head(X.mean(dim=1))  # average over all patches (this exercise)
```

**Effect on accuracy.** The original ViT paper (Dosovitskiy et al., 2021, Appendix D)
found that both heads reach similar accuracy once the learning rate is tuned for each.
On a small dataset trained for few epochs, the averaged head often learns a little faster:
every patch output feeds the loss directly, while the `<cls>` token first has to learn,
through attention, how to collect information from the patches. Several later training
recipes for ViTs on smaller datasets use average pooling by default for this reason.

The cell below trains the averaged model with the same hyperparameters as the `<cls>`
model above and prints both validation accuracies. Differences of about one percentage
point can come from random initialization alone, so repeat with a different `seed`
before drawing a firm conclusion.

In [ ]:
class ViTMeanPool(ViT):
    """Vision Transformer that classifies from the average of the patch outputs."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        del self.cls_token  # not needed when averaging over the patches
        # Keep only the positional embeddings of the patches (drop the <cls> one)
        self.pos_embedding = nn.Parameter(self.pos_embedding.data[:, 1:].clone())

    def forward(self, X):
        X = self.patch_embedding(X)
        X = self.dropout(X + self.pos_embedding)
        for blk in self.blks:
            X = blk(X)
        return self.head(X.mean(dim=1))  # average over the patch dimension


def evaluate_accuracy(net, data):
    """Fraction of correctly classified images in the validation set."""
    net.eval()
    device = next(net.parameters()).device
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in data.val_dataloader():
            X, y = X.to(device), y.to(device)
            correct += (net(X).argmax(dim=1) == y).sum().item()
            total += y.numel()
    return correct / total


hparams = dict(img_size=96, patch_size=16, num_hiddens=512, mlp_num_hiddens=2048,
               num_heads=8, num_blks=2, emb_dropout=0.1, blk_dropout=0.1, lr=0.1)
torch.manual_seed(0)
model_mean = ViTMeanPool(**hparams)
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
trainer.fit(model_mean, data)  # same data as the <cls> model above

acc_cls = evaluate_accuracy(model, data)  # the <cls> model trained above
acc_mean = evaluate_accuracy(model_mean, data)
print(f'<cls> token head:    {acc_cls:.3f}')
print(f'averaged patch head: {acc_mean:.3f}')

### Exercise 3: Tune the hyperparameters to improve accuracy

The default settings leave a lot of accuracy on the table for this dataset:

1. **Optimizer.** Plain SGD with `lr=0.1` and a constant learning rate is a poor fit for
   Transformers, which are sensitive to the learning rate. The standard recipe for ViTs is
   AdamW (Adam with decoupled weight decay) with a learning rate around `1e-3`, weight decay,
   a learning rate that decreases during training, and gradient clipping.
2. **Image and patch size.** As shown in Exercise 1, upscaling 28×28 images to 96×96 only
   adds cost. Working at the native 28×28 with `patch_size=4` gives 49 patches of 4×4 real
   pixels each: slightly more and finer patches than the 36 used now (each about 4.7×4.7
   original pixels), with far less resizing and patch-embedding work.
3. **Depth versus width.** Two wide blocks (`num_hiddens=512`) make a shallow model. For
   small images, a narrower but deeper encoder, such as 6 blocks with `num_hiddens=256`, is
   usually a better use of the same compute.
4. **Training length.** ViTs lack the built-in assumptions of CNNs (locality and translation
   equivariance, see the summary) and need more updates to learn them from data, so train
   for 20 epochs instead of 10.
5. **Regularization.** Longer training raises the risk of overfitting; dropout and weight
   decay counter it.

The cell below applies all of these changes. `ViTAdamW` overrides `configure_optimizers`
to use AdamW, and `CosineTrainer` lowers the learning rate along a cosine curve after
each epoch. To see which change helps most, change one setting at a time
(for example only the optimizer, then only the image and patch size) and compare.
Further options are data augmentation (random crops and horizontal flips) and label smoothing.

In [ ]:
class ViTAdamW(ViT):
    """Vision Transformer trained with AdamW and weight decay instead of SGD."""
    def __init__(self, *args, weight_decay=0.05, **kwargs):
        super().__init__(*args, **kwargs)
        self.weight_decay = weight_decay

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr,
                                 weight_decay=self.weight_decay)


class CosineTrainer(d2l.Trainer):
    """d2l.Trainer that lowers the learning rate along a cosine curve."""
    def fit_epoch(self):
        if self.epoch == 0:
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                self.optim, T_max=self.max_epochs)
        super().fit_epoch()
        self.scheduler.step()


tuned = dict(img_size=28, patch_size=4, num_hiddens=256, mlp_num_hiddens=1024,
             num_heads=8, num_blks=6, emb_dropout=0.1, blk_dropout=0.1, lr=1e-3)
torch.manual_seed(0)
model_tuned = ViTAdamW(**tuned, weight_decay=0.05)
trainer = CosineTrainer(max_epochs=20, num_gpus=1, gradient_clip_val=1.0)
data_tuned = d2l.FashionMNIST(batch_size=128, resize=(28, 28))
trainer.fit(model_tuned, data_tuned)

print(f'baseline (SGD, 96x96, 2 blocks, 10 epochs):   {acc_cls:.3f}')
print(f'tuned (AdamW + cosine, 28x28, 6 blocks, 20 epochs): '
      f'{evaluate_accuracy(model_tuned, data_tuned):.3f}')

[Discussions](https://discuss.d2l.ai/t/8943)
